In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df = pd.read_csv('data/dirty_cafe_sales.csv')
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# print all columns 

pd.options.display.max_columns = None

# print all rows
pd.options.display.max_rows = None

In [ ]:
print(df[df['Item'].isnull()])

In [ ]:
# If dtype didn't change, it's usually because some values can't be parsed as numbers.
# With `errors='ignore'`, pandas silently keeps the original dtype.

def _clean_to_numeric(s: pd.Series) -> pd.Series:
    s = s.astype('string')
    s = s.str.strip()
    # Remove common formatting/currency characters before parsing
    s = s.str.replace(',', '', regex=False)
    s = s.str.replace(r'[^0-9.\-]', '', regex=True)
    return pd.to_numeric(s, errors='coerce')

for col, target in {
    'Quantity': 'Int64',
    'Price Per Unit': 'Float64',
    'Total Spent': 'Float64',
}.items():
    numeric = _clean_to_numeric(df[col])
    failed = numeric.isna() & df[col].notna()
    print(f"{col}: failed to parse {failed.sum()} values")
    df[col] = numeric.astype(target)

df.info()

In [ ]:
df.isnull().sum()

In [ ]:
df.duplicated().sum()

In [ ]:
item_counts=df['Item'].value_counts()
print(item_counts)

In [ ]:
# Normalize Payment Method: map ERROR/UNKNOWN -> Other
pm = df['Payment Method'].astype('string').str.strip()
pm_upper = pm.str.upper()
pm = pm.where(~pm_upper.isin(['ERROR', 'UNKNOWN']), 'Other')
df['Payment Method'] = pm

payment_methods_counts = df['Payment Method'].value_counts()
print(payment_methods_counts)

In [ ]:
# Confusion-matrix style table: Item (rows) vs Location (columns)
pm = df['Item'].fillna('Unknown').astype('string').str.strip()
loc = df['Location'].fillna('Unknown').astype('string').str.strip()

cm = pd.crosstab(pm, loc)
cm

In [ ]:
location_counts = df['Location'].value_counts()
print(location_counts)

# Heatmap view of the confusion matrix (counts)
plt.figure(figsize=(12, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title('Item vs Location (Counts)')
plt.xlabel('Location')
plt.ylabel('Item')
plt.tight_layout()
plt.show()

In [ ]:
df.info()

In [ ]:
# Fill missing values in Quantity / Price Per Unit / Total Spent
cols = ['Quantity', 'Price Per Unit', 'Total Spent']
print('Nulls before:')
print(df[cols].isna().sum())

q = df['Quantity']
p = df['Price Per Unit']
t = df['Total Spent']

# 1) Use the relationship: Total Spent = Quantity * Price Per Unit
mask_t = t.isna() & q.notna() & p.notna()
df.loc[mask_t, 'Total Spent'] = (q * p)[mask_t]

mask_p = p.isna() & t.notna() & q.notna() & (q != 0)
df.loc[mask_p, 'Price Per Unit'] = (t / q)[mask_p]

mask_q = q.isna() & t.notna() & p.notna() & (p != 0)
df.loc[mask_q, 'Quantity'] = (t / p).round()[mask_q]

# 2) Fill anything still missing with simple statistics
quantity_mode = df['Quantity'].dropna().mode()
fill_quantity = int(quantity_mode.iloc[0]) if not quantity_mode.empty else 1
df['Quantity'] = df['Quantity'].fillna(fill_quantity)

df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Price Per Unit'].median())

# If Total Spent still missing, recompute from q*p if possible, else use median
mask_t2 = df['Total Spent'].isna() & df['Quantity'].notna() & df['Price Per Unit'].notna()
df.loc[mask_t2, 'Total Spent'] = (df['Quantity'] * df['Price Per Unit'])[mask_t2]
df['Total Spent'] = df['Total Spent'].fillna(df['Total Spent'].median())

# Keep dtypes nice (nullable)
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce').astype('Int64')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce').astype('Float64')
df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce').astype('Float64')

print('\nNulls after:')
print(df[cols].isna().sum())

df[cols].describe()

In [ ]:
df.info()

In [ ]:
# Convert null/blank Location values to 'Unknown' (also map ERROR/UNKNOWN -> Unknown)
loc = df['Location'].astype('string').str.strip()
loc = loc.replace({'': pd.NA})
loc = loc.fillna('Unknown')

# Normalize common markers
loc_upper = loc.str.upper()
loc = loc.where(~loc_upper.isin(['UNKNOWN', 'ERROR']), 'Unknown')
df['Location'] = loc

print('Missing Location:', int(df['Location'].isna().sum()))
print(df['Location'].value_counts(dropna=False))

In [ ]:
# Convert missing/invalid Payment Method values to 'Other'
pm = df['Payment Method'].astype('string').str.strip()
pm_upper = pm.str.upper()

# Map known bad markers to Other
pm = pm.where(~pm_upper.isin(['ERROR', 'UNKNOWN']), 'Other')

# Map missing/blank to Other
pm = pm.fillna('Other')
pm = pm.replace({'': 'Other'})

df['Payment Method'] = pm

print(df['Payment Method'].value_counts(dropna=False))

In [ ]:
# Clean + fill unknown Item values using price (do this BEFORE location splitting)
item = df['Item'].astype('string').str.strip()
item_upper = item.str.upper()

# Treat placeholders like missing
item = item.where(~item_upper.isin(['UNKNOWN', 'ERROR', '']), pd.NA)
df['Item'] = item

# Make sure price is numeric (bad strings -> NaN)
price = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['price_key'] = price.round(2)

print('Missing/unknown Item before fill:', int(df['Item'].isna().sum()))

# For each price, pick the most common Item and how confident we are
known = df[df['Item'].notna() & df['price_key'].notna()]
most_common_item = known.groupby('price_key')['Item'].agg(lambda x: x.value_counts().idxmax())
most_common_share = known.groupby('price_key')['Item'].agg(lambda x: x.value_counts(normalize=True).max())

threshold = 0.80
mask = df['Item'].isna() & df['price_key'].notna()
prices = df.loc[mask, 'price_key']
fill_values = prices.map(most_common_item)
confident = prices.map(most_common_share) >= threshold

df.loc[prices.index[confident], 'Item'] = fill_values[confident]

print('Filled Items (price-based):', int(confident.sum()))
print('Missing/unknown Item after fill:', int(df['Item'].isna().sum()))

# Quick look: which item each price usually means
mapping = pd.DataFrame({'Most Common Item': most_common_item, 'Share': most_common_share}).sort_index()
display(mapping)

In [ ]:
# Probabilistic imputation: assign Unknown Location -> Takeaway/In-store based on Item
import numpy as np

# Use only the 2 real locations to learn probabilities
known_mask = df['Location'].isin(['Takeaway', 'In-store']) & df['Item'].notna()
learn = df.loc[known_mask, ['Item', 'Location']].copy()

# Count Item by Location (only Takeaway/In-store)
counts = pd.crosstab(learn['Item'], learn['Location'])
counts = counts.reindex(columns=['Takeaway', 'In-store'], fill_value=0)

# Show items ordered in BOTH locations
both = counts[(counts['Takeaway'] > 0) & (counts['In-store'] > 0)].copy()
both['Total'] = both['Takeaway'] + both['In-store']
both['P(Takeaway|Item)'] = both['Takeaway'] / both['Total']
both['P(In-store|Item)'] = both['In-store'] / both['Total']
print('Items ordered in both Takeaway & In-store (top 10 by total orders):')
display(both.sort_values('Total', ascending=False).head(10))

# Probability tables for sampling by item
prob_takeaway = (counts['Takeaway'] / counts.sum(axis=1)).to_dict()
prob_instore = (counts['In-store'] / counts.sum(axis=1)).to_dict()

# Overall fallback probability (if Item is missing or never seen)
overall = df.loc[df['Location'].isin(['Takeaway', 'In-store']), 'Location'].value_counts(normalize=True)
p_takeaway_overall = float(overall.get('Takeaway', 0.5))
p_instore_overall = float(overall.get('In-store', 0.5))

unknown_before = int((df['Location'] == 'Unknown').sum())

np.random.seed(42)  # reproducible random assignment

# 1) Fill Unknown locations when Item is present (use P(Location|Item))
unknown_with_item = df['Location'].eq('Unknown') & df['Item'].notna()
choices = []
for item in df.loc[unknown_with_item, 'Item']:
    p_t = prob_takeaway.get(item, p_takeaway_overall)
    p_i = prob_instore.get(item, p_instore_overall)
    s = p_t + p_i
    if s == 0:
        p_t, p_i = p_takeaway_overall, p_instore_overall
        s = p_t + p_i
    p_t, p_i = p_t / s, p_i / s
    choices.append(np.random.choice(['Takeaway', 'In-store'], p=[p_t, p_i]))
df.loc[unknown_with_item, 'Location'] = choices

# 2) Fill any remaining Unknown locations (Item missing) using overall proportions
still_unknown = df['Location'].eq('Unknown')
unknown_without_item = still_unknown & df['Item'].isna()
choices2 = np.random.choice(['Takeaway', 'In-store'], size=int(unknown_without_item.sum()), p=[p_takeaway_overall, p_instore_overall])
df.loc[unknown_without_item, 'Location'] = choices2

print('\nUnknown Location before:', unknown_before)
print('Imputed using Item:', int(unknown_with_item.sum()))
print('Imputed using overall split (no Item):', int(unknown_without_item.sum()))
print('Unknown Location after:', int((df["Location"] == "Unknown").sum()))
print('\nLocation counts after imputation:')
print(df['Location'].value_counts())

In [ ]:
df.info()

In [ ]:
from __future__ import annotations



from pathlib import Path

import json



import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

import seaborn as sns



# --- Build BEFORE vs AFTER artifacts for the report ---

out_dir = Path("data/assets/cafe")

out_dir.mkdir(parents=True, exist_ok=True)





# Before: raw file

df_before = pd.read_csv("data/dirty_cafe_sales.csv")



# After: cleaned dataframe produced by this notebook

df_after = df.copy()



def _missing_counts(frame: pd.DataFrame) -> pd.Series:

    return frame.isna().sum().sort_values(ascending=False)



def _coerce_numeric(series: pd.Series) -> pd.Series:

    s = series.astype(str).str.replace(r"[^0-9.\-]", "", regex=True)

    s = s.replace({"": np.nan, "nan": np.nan, "None": np.nan})

    return pd.to_numeric(s, errors="coerce")



missing_before = _missing_counts(df_before)

missing_after = _missing_counts(df_after)



# 1) Missing values per column

missing_plot = pd.DataFrame(

    {

        "Before": missing_before,

        "After": missing_after.reindex(missing_before.index, fill_value=0),

    }

)

fig, ax = plt.subplots(figsize=(12, 4))

missing_plot.plot(kind="bar", ax=ax)

ax.set_title("Missing values per column (before vs after)")

ax.set_ylabel("Count")

ax.set_xlabel("Column")

ax.legend(loc="upper right")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()

fig.savefig(out_dir / "missingness_before_after.png", dpi=200)

plt.close(fig)



# 2) Numeric distributions

num_cols = ["Quantity", "Price Per Unit", "Total Spent"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for i, col in enumerate(num_cols):

    before_num = _coerce_numeric(df_before[col]) if col in df_before.columns else pd.Series(dtype=float)

    after_num = pd.to_numeric(df_after[col], errors="coerce") if col in df_after.columns else pd.Series(dtype=float)



    sns.histplot(before_num, ax=axes[i], bins=30, stat="density", kde=True, alpha=0.35, color="tab:orange", label="Before")

    sns.histplot(after_num, ax=axes[i], bins=30, stat="density", kde=True, alpha=0.35, color="tab:blue", label="After")

    axes[i].set_title(col)

    axes[i].set_ylabel("Density")

    axes[i].legend()

plt.tight_layout()

fig.savefig(out_dir / "numeric_distributions_before_after.png", dpi=200)

plt.close(fig)



# 3) Top categories (Item, Payment Method, Location)

cat_cols = ["Item", "Payment Method", "Location"]

fig, axes = plt.subplots(1, 3, figsize=(18, 4))

for i, col in enumerate(cat_cols):

    before_counts = df_before[col].fillna("(missing)").astype(str).value_counts().head(7)

    after_counts = df_after[col].fillna("(missing)").astype(str).value_counts().head(7)

    top = pd.DataFrame({"Before": before_counts, "After": after_counts}).fillna(0)

    top.plot(kind="bar", ax=axes[i])

    axes[i].set_title(f"Top values: {col}")

    axes[i].set_xlabel("")

    axes[i].set_ylabel("Count")

    axes[i].tick_params(axis="x", rotation=45)

plt.tight_layout()

fig.savefig(out_dir / "top_categories_before_after.png", dpi=200)

plt.close(fig)



def _series_to_int_dict(s: pd.Series) -> dict[str, int]:

    return {str(k): int(v) for k, v in s.items()}



metrics = {

    "dataset": "cafe",

    "before": {

        "shape": [int(df_before.shape[0]), int(df_before.shape[1])],

        "missing_total": int(df_before.isna().sum().sum()),

        "missing_by_col": _series_to_int_dict(missing_before),

        "duplicates": int(df_before.duplicated().sum()),

    },

    "after": {

        "shape": [int(df_after.shape[0]), int(df_after.shape[1])],

        "missing_total": int(df_after.isna().sum().sum()),

        "missing_by_col": _series_to_int_dict(missing_after),

        "duplicates": int(df_after.duplicated().sum()),

    },

}



print("Saved cafe plots ->", out_dir)
